# Step 3: input diagnostics on the full dataset: what does the ViT actually see? (CPU, ~15 min)
1. Session options: Accelerator **None**, Internet **ON**, secret `GITHUB_TOKEN` (or attach the code Dataset).
2. **+ Add Input**: `mnj-cache2-ispy1`, `mnj-cache2-ispy2`, `mnj-cache2-duke` (from step 2). The team's v1 caches
   `mnj-cache-*` also work, and attaching both lets you compare.

Measures, on **training patients only**:
- **Saturation:** how much of the tumour is clipped to white by the v1 joint 99.5th-percentile scaling, and by
  each alternative normalisation (v2 caches are exact; on v1 the clipped part is already gone).
- **Tumour size in the input:** the share of the 224 px input width that the tumour box takes, per crop.
- **Gallery:** the same slices under every normalisation, pCR vs non-pCR.
- **Is there signal at all?** Hand-crafted tumour enhancement features (no deep learning) -> logistic regression:
  5-fold CV AUC on train and AUC on val. The `cohort_only` row is the AUC you get for free from cohort pCR rates.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # "A", "B" or "C" (your role in the team): results go to results/<PERSON>/
N_PER_COHORT = 80        # training patients per cohort for the saturation / size statistics
PREFER = "v2"            # cache format to use when both are attached

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
assert PERSON in ("A", "B", "C"), 'set PERSON = "A", "B" or "C" in the parameters cell'
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob
    hits = [h for d in range(1, 6) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    assert hits, ("no GITHUB_TOKEN secret and no code Dataset attached. Either add the secret (Add-ons -> Secrets) "
                  "or attach the Dataset made from mnj-code-vaib.zip (see kaggle/vaib/README.md)")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from src.data.metadata import patient_table
from src.vitonly.slabs import find_caches, load_slab
from src.vitonly.norms import NORMS, CROPS
from src.vitonly.diagnostics import saturation, v1_clipped, tumor_fraction, gallery
from src.vitonly.kinetics import kinetic_features, FEATURES
from src.vitonly.probe import score
caches = find_caches(["/kaggle/input", "/kaggle/working"], PREFER)
fmts = {c: json.load(open(Path(d) / "cache_meta.json")).get("format", "v1") for c, d in caches.items()}
print({c: (d, fmts[c]) for c, d in caches.items()})
cohorts = [c for c in ("ispy1", "ispy2", "duke") if c in caches]
train = patient_table(["train"], cohorts, cache_dirs=caches)
val = patient_table(["val"], cohorts, cache_dirs=caches)
print(f"train {len(train)}, val {len(val)}")

## 1. Saturation inside the tumour and tumour size in the input

In [ ]:
rows = []
for c in cohorts:
    for r in train[train.cohort == c].sample(min(N_PER_COHORT, (train.cohort == c).sum()), random_state=0).itertuples():
        s = load_slab(Path(caches[c]) / f"{r.pid}.npz")
        sat = saturation(s)
        rows.append({"cohort": c, "pid": r.pid, "pCR": r.pCR, "v1_clipped": float(v1_clipped(s)[1:].max()),
                     **{n: float(sat[n][1:].max()) for n in NORMS},
                     **{f"size_{k}": v for k, v in tumor_fraction(s, r.xy_spacing).items()}})
diag = pd.DataFrame(rows)
print("fraction of tumour pixels saturated (worst of early/late channel), mean per cohort; cache:", fmts)
display(diag.groupby("cohort")[["v1_clipped", *NORMS]].mean().round(3))
print("share of the 224 px input width taken by the tumour box, mean per cohort")
display(diag.groupby("cohort")[[f"size_{k}" for k in CROPS]].mean().round(2))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
diag.groupby("cohort")[["v1_clipped", *NORMS]].mean().T.plot.bar(ax=ax[0]); ax[0].set_title("tumour pixels saturated")
diag.groupby("cohort")[[f"size_{k}" for k in CROPS]].mean().T.plot.bar(ax=ax[1]); ax[1].set_title("tumour share of input width")
plt.tight_layout(); plt.show()
if "v1" in fmts.values():
    print("NOTE: on v1 caches only `v1_clipped` measures the loss; other columns can't see what v1 already clipped.")

## 2. Gallery: the same middle-tumour slice under every normalisation (tumour crop, mask in white)

In [ ]:
for c in cohorts:
    t = train[train.cohort == c]
    pick = pd.concat([g.sample(min(2, len(g)), random_state=1) for _, g in t.groupby("pCR")])
    fig, axes = plt.subplots(len(pick), len(NORMS), figsize=(1.9 * len(NORMS), 2.0 * len(pick)), squeeze=False)
    for row, r in zip(axes, pick.itertuples()):
        gallery(row, load_slab(Path(caches[c]) / f"{r.pid}.npz"), r.xy_spacing, NORMS, "tumor")
        row[0].set_title(f"{r.pid} pCR={int(r.pCR)}", fontsize=7, loc="left")
    for a, n in zip(axes[-1], NORMS): a.set_xlabel(n); a.axis("on"); a.set_xticks([]); a.set_yticks([])
    plt.suptitle(f"{c} ({fmts[c]} cache): " + " | ".join(NORMS), fontsize=8); plt.tight_layout(); plt.show()

## 3. Is there pCR signal in the images without deep learning?
Tumour ROI = voxels in the tumour box with percent enhancement >= 70% (the I-SPY functional tumour volume rule),
the same rule for every cohort. Features: functional tumour volume, PE and SER statistics, washout/plateau/
persistent fractions, relative intensities (16 in total).

In [ ]:
table = pd.concat([train, val]).reset_index(drop=True)
is_train = np.r_[np.ones(len(train), bool), np.zeros(len(val), bool)]
X = np.stack([kinetic_features(load_slab(Path(caches[r.cohort]) / f"{r.pid}.npz"), r.xy_spacing, r.slice_thick)
              for r in table.itertuples()])
y, coh = table.pCR.to_numpy(float), table.cohort.to_numpy(str)
res = {}
for name, F in (("kinetic_handcrafted", np.nan_to_num(X)), ("cohort_only", pd.get_dummies(pd.Series(coh)).to_numpy(float)),
                ("kinetic + cohort", np.c_[np.nan_to_num(X), pd.get_dummies(pd.Series(coh)).to_numpy(float)])):
    res[name] = score(F, y, coh, is_train)[0]
display(pd.DataFrame(res).T[["cv_auc", "cv_auc_sd", "val_auc", "val_ci_lo", "val_ci_hi", "val_auc_within",
                             *[f"val_auc_{c}" for c in cohorts]]].round(3))
from sklearn.metrics import roc_auc_score
tr = is_train
print("single-feature AUC on train (>0.5: higher in pCR):")
display(pd.Series({f: roc_auc_score(y[tr], np.nan_to_num(X[tr, j])) for j, f in enumerate(FEATURES)}).round(3).sort_values())
print("Reference: the old B3 ViT reached val AUC 0.530; clinical-only LR (B0) 0.639.")